# Sensitivity Analysis to K

In [1]:
import itertools
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

import sensitivity as sens

EVAL_DIR = Path.cwd()
FIGURES_DIR = EVAL_DIR / "figures"
FIGURES_DIR.mkdir(exist_ok=True)

MODELS = [("cqd", "CQD"), ("betae", "BetaE"), ("query2box", "Query2Box"), ("gqe", "GQE")]
QUERY_TYPES = [
    ("2p", "2p"), ("3p", "3p"),
    ("2i", "2i"), ("3i", "3i"),
    ("2u", "2u"), ("up", "2u1p"),
    ("ip", "1p2i"), ("pi", "2i1p"),
]
QTYPE_DISPLAY = dict(QUERY_TYPES)
DATASETS = [("FB", "FB15k-237+H"), ("NELL", "NELL995+H")]

METRICS = ["Necessity", "Sufficiency"]
METRIC_DISPLAY = {"Necessity": "<b>Necessary</b>", "Sufficiency": "<b>Sufficient</b>"}
METRIC_LABEL = {"Necessity": "Necc", "Sufficiency": "Suff"}

K_TABLE = [1, 3, 5, 7, 10]        # curated K's used in every table
K_VALUES_DOTPLOT = [1, 3, 5, 10]  # markers shown in the dot figure
K_VALUES_FULL = list(range(1, 11))  # full sweep for the disagreement line figure

DRAWIO = {
    "blue":    {"fill": "#B0E3E6", "line": "#0E8088"},
    "orange":  {"fill": "#FAD7AC", "line": "#B46504"},
    "purple":  {"fill": "#D0CEE2", "line": "#56517E"},
    "red":     {"fill": "#FAD9D5", "line": "#AE4132"},
    "green":   {"fill": "#D5E8D4", "line": "#82B366"},
    "yellow":  {"fill": "#FFF2CC", "line": "#D6B656"},
    "gray":    {"fill": "#F5F5F5", "line": "#666666"},
    "magenta": {"fill": "#F0D0E5", "line": "#A3467A"},
}
K_COLORS = {1: DRAWIO["blue"], 3: DRAWIO["orange"], 5: DRAWIO["purple"], 10: DRAWIO["red"]}
K_MARKERS = {1: "circle", 3: "triangle-up", 5: "square", 10: "star"}
QTYPE_COLOR_ORDER = ["blue", "orange", "purple", "red", "green", "yellow", "gray", "magenta"]
QTYPE_COLORS = {q: DRAWIO[QTYPE_COLOR_ORDER[i]] for i, (q, _) in enumerate(QUERY_TYPES)}

FONT_FAMILY = "Times New Roman, Times, serif"

# acmart sigconf geometry (paperwidth=8.5in=612pt, inner=outer=54pt margins,
# columnsep=24pt): \textwidth = 612-2*54 = 504pt, \columnwidth = (504-24)/2 = 240pt.
# Kaleido/plotly export width/height in CSS px; 1 CSS px = 0.75pt in the
# final PDF. To avoid \includegraphics silently rescaling (and shrinking)
# our fonts, we export each figure at EXACTLY its final \textwidth in pt
# (converted to px), so what we set as a plotly font size (px) times 0.75
# is the literal size it prints at -- no hidden LaTeX-side scaling.
TEXTWIDTH_PT = 504.0
PT_PER_PX = 0.75
FIG_TEXTWIDTH_PX = TEXTWIDTH_PT / PT_PER_PX  # 672px

In [2]:
file_map = {}
for ds, _ in DATASETS:
    for model, _ in MODELS:
        for qsuffix, _ in QUERY_TYPES:
            p = EVAL_DIR / f"evaluation_{ds}_2_{model}_{qsuffix}_random.csv"
            if p.exists():
                file_map[(ds, model, qsuffix)] = p

active_models = [(m, d) for m, d in MODELS if any(k[1] == m for k in file_map)]

def model_spokes(dataset, model):
    return [q for q, _ in QUERY_TYPES if (dataset, model, q) in file_map]

print(f"{len(file_map)} result files found for {len(active_models)} models x {len(DATASETS)} datasets")

64 result files found for 4 models x 2 datasets


In [3]:
CACHE_DELTAMRR = EVAL_DIR / "sensitivity_deltamrr_both.csv"
CACHE_DISAGREEMENT = EVAL_DIR / "sensitivity_disagreement_both.csv"
CACHE_CASES = EVAL_DIR / "sensitivity_margin_cases_both.csv"

if CACHE_DELTAMRR.exists() and CACHE_DISAGREEMENT.exists() and CACHE_CASES.exists():
    deltamrr_df = pd.read_csv(CACHE_DELTAMRR)
    disagree_df = pd.read_csv(CACHE_DISAGREEMENT)
    case_df = pd.read_csv(CACHE_CASES)
    print(f"Loaded cached CSVs ({len(deltamrr_df)}, {len(disagree_df)}, {len(case_df)} rows)")
else:
    deltamrr_rows, disagree_rows, case_rows = [], [], []
    for (ds, model, q), path in file_map.items():
        sweep = sens.exhaustive_k_sweep(str(path), K_TABLE)
        sweep.insert(0, "dataset", ds); sweep.insert(2, "model", model); sweep.insert(3, "query_type", q)
        deltamrr_rows.append(sweep)

        rate_df, cases = sens.exhaustive_atom_disagreement_sweep(str(path), K_VALUES_FULL)
        rate_df.insert(0, "dataset", ds); rate_df.insert(2, "model", model); rate_df.insert(3, "query_type", q)
        disagree_rows.append(rate_df)

        cases.insert(0, "dataset", ds); cases.insert(1, "model", model)
        case_rows.append(cases)
        print(f"  done {ds} {model} {q}")

    deltamrr_df = pd.concat(deltamrr_rows, ignore_index=True)
    disagree_df = pd.concat(disagree_rows, ignore_index=True)
    case_df = pd.concat(case_rows, ignore_index=True)
    deltamrr_df.to_csv(CACHE_DELTAMRR, index=False)
    disagree_df.to_csv(CACHE_DISAGREEMENT, index=False)
    case_df.to_csv(CACHE_CASES, index=False)
    print(f"Computed and cached {len(deltamrr_df)}, {len(disagree_df)}, {len(case_df)} rows")

Loaded cached CSVs (640, 640, 1760000 rows)


In [9]:
MODEL_COLORS = {
    "cqd": DRAWIO["blue"], "betae": DRAWIO["orange"],
    "query2box": DRAWIO["purple"], "gqe": DRAWIO["red"],
}

D_TICK_PT, D_TITLE_PT, D_ANNOT_PT = 12.0, 13.0, 14.0
d_tickfont_px = D_TICK_PT / PT_PER_PX
d_titlefont_px = D_TITLE_PT / PT_PER_PX
d_annotfont_px = D_ANNOT_PT / PT_PER_PX

piv_d = deltamrr_df.pivot_table(index=["dataset", "model", "query_type", "metric"], columns="K", values="mean")
dev_d = piv_d.sub(piv_d[10], axis=0).abs()
disagree_tbl_d = disagree_df[disagree_df["K"].isin(K_TABLE)]


def model_avg_deltamrr(dataset, metric, model):
    vals_by_k = []
    for K in K_TABLE:
        vals = [dev_d.loc[(dataset, model, q, metric), K] for q, _ in QUERY_TYPES
                 if (dataset, model, q, metric) in dev_d.index]
        vals_by_k.append(np.mean(vals))
    return vals_by_k


def model_avg_disagreement(dataset, model):
    sub = disagree_tbl_d[(disagree_tbl_d["dataset"] == dataset) & (disagree_tbl_d["model"] == model)]
    return [sub[sub["K"] == K]["mean_disagreement"].mean() * 100.0 for K in K_TABLE]


def model_line_axis_config(ytitle=None, ysuffix="%"):
    return dict(
        xaxis=dict(tickvals=K_TABLE, tickangle=0, tickfont=dict(size=d_tickfont_px, family=FONT_FAMILY),
                   gridcolor="#e3e2dc"),
        yaxis=dict(tickfont=dict(size=d_tickfont_px, family=FONT_FAMILY),
                   ticksuffix=ysuffix, gridcolor="#e3e2dc", rangemode="tozero",
                   title=dict(text=ytitle, font=dict(size=d_titlefont_px, family=FONT_FAMILY), standoff=4)),
    )


ROW_SPECS_D = [
    ("necc", "Necessary", "Deviation (pp)"),
    ("suff", "Sufficient", "Deviation (pp)"),
    ("dis", "Disagreement", "Disagreement (%)"),
]

fig_d = make_subplots(
    rows=3, cols=2,
    column_titles=["<b>FB15k-237+H</b>", "<b>NELL995+H</b>"],
    row_titles=[f"<b>{label}</b>" for _, label, _ in ROW_SPECS_D],
    horizontal_spacing=0.06, vertical_spacing=0.04,
)

for r, (row_key, row_label, ytitle) in enumerate(ROW_SPECS_D, start=1):
    for c, (ds, ds_display) in enumerate(DATASETS, start=1):
        idx = (r - 1) * 2 + c
        for model, mdisp in MODELS:
            if row_key == "necc":
                y = model_avg_deltamrr(ds, "Necessity", model)
            elif row_key == "suff":
                y = model_avg_deltamrr(ds, "Sufficiency", model)
            else:
                y = model_avg_disagreement(ds, model)
            fig_d.add_trace(go.Scatter(
                x=K_TABLE, y=y, mode="lines+markers", name=mdisp,
                legendgroup=model, showlegend=(r == 1 and c == 1),
                line=dict(color=MODEL_COLORS[model]["line"], width=2),
                marker=dict(size=6, color=MODEL_COLORS[model]["line"]),
                hovertemplate=f"{mdisp}<br>K=%{{x}}<br>%{{y:.2f}}<extra></extra>",
            ), row=r, col=c)
        xk, yk = cart_axis_keys(idx)
        cfg = model_line_axis_config(ytitle=ytitle if c == 1 else None, ysuffix="")
        fig_d.update_layout(**{xk: cfg["xaxis"], yk: cfg["yaxis"]})

fig_d.update_annotations(font=dict(size=d_annotfont_px, family=FONT_FAMILY))
fig_d.update_layout(
    template="plotly_white",
    font=dict(size=d_tickfont_px, family=FONT_FAMILY),
    legend=dict(orientation="h", font=dict(size=d_titlefont_px, family=FONT_FAMILY),
                x=0.5, xanchor="center", y=1.06, yanchor="bottom"),
    width=FIG_TEXTWIDTH_PX, height=460 / PT_PER_PX,
    margin=dict(t=60, b=30, l=55, r=15),
)
fig_d.show()

out = FIGURES_DIR / "sensitivity_main_combined"
fig_d.write_image(str(out) + ".pdf", width=FIG_TEXTWIDTH_PX, height=460 / PT_PER_PX)
fig_d.write_image(str(out) + ".png", width=FIG_TEXTWIDTH_PX, height=460 / PT_PER_PX, scale=2)
print(f"Saved {out.name}.{{pdf,png}} at native {TEXTWIDTH_PT}x460pt")

Saved sensitivity_main_combined.{pdf,png} at native 504.0x460pt
